In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np
from rdkit.Chem import rdDistGeom
from rdkit.Chem.rdMolTransforms import SetDihedralDeg
from torch_geometric.data import Data
import torch

In [ ]:
def create_conformers(mol, num_conformers=5):
    mol_list = []

    def get_dihedral_angles_atoms(mol):
            rotatable_bonds_list = []
            dihedral_angles_list = []

            def find_rotatable_bonds():
                rotatable_ring_bonds = Chem.MolFromSmarts("[!$(*#*)&X3&R]-!@[!$(*#*)&X3&R]")
                rotatable_matches = mol.GetSubstructMatches(rotatable_ring_bonds)

                for bond in rotatable_matches:
                    if bond[0] < bond[1]:
                        rotatable_bonds_list.append((bond[0], bond[1]))
                    else:
                        rotatable_bonds_list.append((bond[1], bond[0]))

            def get_torsion():
                for bond in rotatable_bonds_list:
                    four_atom_bond = []

                    for atom in mol.GetAtomWithIdx(bond[0]).GetNeighbors():
                        idx = atom.GetIdx()

                        if idx != bond[1]:
                            four_atom_bond.append(idx)
                            four_atom_bond.append(bond[0])
                            break
                    for atom in mol.GetAtomWithIdx(bond[1]).GetNeighbors():
                        idx = atom.GetIdx()

                        if idx != bond[0]:
                            four_atom_bond.append(bond[1])
                            four_atom_bond.append(idx)
                            break

                    dihedral_angles_list.append(four_atom_bond)

            find_rotatable_bonds()
            get_torsion()

            return dihedral_angles_list

    def set_angles_deg(conf, dihedral_angles_list, original_angles_deg):
        for atoms, angle in zip(dihedral_angles_list, original_angles_deg):
            SetDihedralDeg(conf, atoms[0], atoms[1], atoms[2], atoms[3], float(angle))

    def rand_deg(n_ang):
        flag = True
        while flag == True:
            rand = np.random.choice([0, 180], size=(num_conformers, n_ang))
            if len(np.unique(rand, axis=0)) != num_conformers:
                rand = np.random.choice([0, 180], size=(num_conformers, n_ang))
            else: flag = False

        return rand

    dihedral_angles_list = get_dihedral_angles_atoms(mol)

    dihedral_angles_list_len = len(dihedral_angles_list)

    max_possible_conformers = 2 ** dihedral_angles_list_len
    num_conformers = min(num_conformers, max_possible_conformers)

    rand_angles_deg = rand_deg(dihedral_angles_list_len)

    molh = Chem.AddHs(mol)

    for i in range(num_conformers):
        new_mol = Chem.Mol(molh)

        if new_mol.GetNumConformers() == 0:
            rdDistGeom.EmbedMolecule(new_mol, randomSeed=42, maxAttempts=100, useRandomCoords=True)

        conf = new_mol.GetConformer()

        angles_list = [float(angle) for angle in rand_angles_deg[i]]

        set_angles_deg(conf, dihedral_angles_list, angles_list)

        AllChem.MMFFOptimizeMolecule(new_mol, mmffVariant='MMFF94s')

        mol_list.append(new_mol)

    return mol_list

In [ ]:
from sklearn.model_selection import train_test_split

def load_data_list(X, y, train_size=0.75, val_size=0.15):
    val_size = val_size/(1-train_size)

    smiles_train, smiles_val_test, y_train, y_val_test = train_test_split(X, y, train_size=train_size, random_state=42)
    smiles_val, smiles_test, y_val, y_test = train_test_split(smiles_val_test, y_val_test, train_size=val_size, random_state=42)

    return [smiles_train, smiles_val, smiles_test, y_train, y_val, y_test]

In [ ]:
data_raw = pd.read_csv(r'C:\Users\AI1-PC\PycharmProjects\main\datasets\polymers\DOI 10.1021acs.jpclett.8b00635_exp.txt', sep=',', index_col=None)
data_no_dub = data_raw.drop_duplicates(subset=['SMILES'])

homo_list_exp = data_no_dub['-HOMO (eV)'].to_numpy()
lumo_list_exp = data_no_dub['-LUMO (eV)'].to_numpy()
gap_list_exp = data_no_dub['bandgap(eV)'].to_numpy()
smiles_list_exp = data_no_dub['SMILES'].to_numpy()

gap_homo_lumo_array = np.array([[gap_list_exp], [homo_list_exp], [lumo_list_exp]]).T.reshape(-1, 3)

In [ ]:
smiles_train, smiles_val, smiles_test, y_train, y_val, y_test = load_data_list(smiles_list_exp, gap_homo_lumo_array, train_size=0.80, val_size=0.10)

In [ ]:
file_number = 0
for smiles, y in zip(smiles_train, y_train):
    mol = Chem.MolFromSmiles(smiles)
    mol_list = create_conformers(mol)

    for m in mol_list:
        x = []
        pos = []

        for i, atom in enumerate(m.GetAtoms()):
            x.append(atom.GetAtomicNum())
            positions = m.GetConformer().GetAtomPosition(i)
            pos.append([positions.x, positions.y, positions.z])

        data = Data(
            x=torch.tensor(x, dtype=torch.long),
            pos=torch.tensor(pos, dtype=torch.float32),
            y=torch.tensor([[y[0], y[1], y[2]]], dtype=torch.float32)
        )

        torch.save(data, fr'C:\Users\AI1-PC\PycharmProjects\main\datasets\polymers\poly_exp_geom_to_prop_augment_train_80_v_1\{file_number}.pt')
        print(file_number)
        file_number = file_number + 1

In [ ]:
file_number = 0
for smiles, y in zip(smiles_val, y_val):
    mol = Chem.MolFromSmiles(smiles)
    molh = Chem.AddHs(mol)

    rdDistGeom.EmbedMolecule(molh, randomSeed=42, maxAttempts=100, useRandomCoords=True)
    AllChem.MMFFOptimizeMolecule(molh, mmffVariant='MMFF94s')

    x = []
    pos = []

    for i, atom in enumerate(molh.GetAtoms()):
        x.append(atom.GetAtomicNum())
        positions = molh.GetConformer().GetAtomPosition(i)
        pos.append([positions.x, positions.y, positions.z])

    data = Data(
        x=torch.tensor(x, dtype=torch.long),
        pos=torch.tensor(pos, dtype=torch.float32),
        y=torch.tensor([[y[0], y[1], y[2]]], dtype=torch.float32)
    )

    torch.save(data, fr'C:\Users\AI1-PC\PycharmProjects\main\datasets\polymers\poly_exp_geom_to_prop_augment_val_10_v_1\{file_number}.pt')
    print(file_number)
    file_number = file_number + 1

In [ ]:
file_number = 0
for smiles, y in zip(smiles_test, y_test):
    mol = Chem.MolFromSmiles(smiles)
    molh = Chem.AddHs(mol)

    rdDistGeom.EmbedMolecule(molh, randomSeed=42, maxAttempts=100, useRandomCoords=True)
    AllChem.MMFFOptimizeMolecule(molh, mmffVariant='MMFF94s')

    x = []
    pos = []

    for i, atom in enumerate(molh.GetAtoms()):
        x.append(atom.GetAtomicNum())
        positions = molh.GetConformer().GetAtomPosition(i)
        pos.append([positions.x, positions.y, positions.z])

    data = Data(
        x=torch.tensor(x, dtype=torch.long),
        pos=torch.tensor(pos, dtype=torch.float32),
        y=torch.tensor([[y[0], y[1], y[2]]], dtype=torch.float32)
    )

    torch.save(data, fr'C:\Users\AI1-PC\PycharmProjects\main\datasets\polymers\poly_exp_geom_to_prop_augment_test_10_v_1\{file_number}.pt')
    print(file_number)
    file_number = file_number + 1